<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


<h1>实验：使用 Haar 级联分类器检测车辆</h1>


## 概述


在本实验中，你将把预训练的 Haar 级联分类器应用于上传的车辆图像以进行车辆检测。


# 目标


Haar 级联是一种基于 Haar 小波识别图像或视频中对象的机器学习方法。我们将使用 [OpenCV](http://opencv.org/) 库。它基于 Paul Viola 和 Michael Jones 在其论文 "Rapid Object Detection using a Boosted Cascade of Simple Features" 中提出的特征概念。


## 目录


本笔记本分为以下几个部分：
        <ul>
            <li>[安装与导入库](#Install-and-Import-Libraries) </li>
            <li>[图像处理](#Image-Processing)</li>
            <li>[练习 - 上传你的图像](#Practice-Exercise---Upload-your-image) </li>
            </li>      
        </ul>
    </li>
</ul>


## 安装与导入库


In [ ]:
!pip install opencv-python-headless
!pip install numpy pandas matplotlib  --quiet


**导入重要库并定义辅助函数**


In [ ]:
import urllib.request
import cv2
from matplotlib import pyplot as plt
%matplotlib inline


**创建一个用于清理和显示图像的函数：**

`plt_show()` 函数是一个实用工具，用于在 Jupyter 笔记本或脚本中使用 matplotlib 可视化图像。

它支持彩色和灰度显示格式。默认情况下，OpenCV 以 BGR 颜色格式加载图像，需要转换为 RGB 才能在 matplotlib 中正确显示。当 gray=False 时，该函数会处理此转换。

它还允许添加标题和自定义显示大小。gray=True 标志用于以灰度查看图像，这在进行边缘检测、目标检测或检查单通道数据（如掩码或特征）时非常有用。该工具在计算机视觉任务的数据预处理、模型评估和调试步骤中特别有用。


In [ ]:
# 使用 matplotlib 显示图像的函数
def plt_show(image, title="", gray=False, size=(10, 10)):
    temp = image.copy()  # 复制一份以避免修改原始图像

    if not gray:
        # 将图像从 BGR（OpenCV 默认格式）转换为 RGB（matplotlib 所需格式）
        temp = cv2.cvtColor(temp, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=size)  # 设置图像大小
    plt.imshow(temp, cmap='gray' if gray else None)  # 显示图像；若 gray=True 则以灰度显示
    plt.title(title)  # 设置图像上方的标题
    plt.axis("off")  # 隐藏坐标轴刻度和标签以获得更简洁的显示
    plt.show()  # 渲染图像


## 🔍 技术原理：`detectMultiScale()` 的检测流程

对每张输入图像，检测器内部按以下流程运行：

1. **转灰度图**：Haar 特征是亮度差特征，只需单通道，所以 `detect_obj()` 先把图像转成灰度图。
2. **积分图加速**：先计算积分图（integral image），之后任意矩形的像素和只需 4 次加减运算即可得到——这是 Haar 检测能实时运行的关键。
3. **滑动窗口 + 多尺度扫描**：用一个固定大小的窗口（尺寸由 XML 决定，约 20×20）在图像上逐位置滑动；同时按 `scaleFactor`（本实验为 1.05）逐级缩小图像再扫一遍，从而检测不同大小的车辆。
4. **级联过滤（核心）**：窗口每停在一个位置，就过一遍级联——第 1 级用最简单、区分度最高的特征快速淘汰绝大多数窗口（先粗筛），通过了才进入第 2、3… 级（再细验）。全部级联都通过的窗口才被认定为「此处有车」，输出一个 `(x, y, w, h)` 边界框。这就是「cascade」的含义：**先粗筛、再细验**，让绝大部分计算只花在少量候选窗口上。
5. **合并重叠框**：同一辆车常常在不同位置/尺度被重复检出多个框，`minNeighbors`（本实验为 2）控制合并阈值——要求一个框周围至少有这么多「邻居」框才保留。值越低越灵敏、误检也越多。
6. **后处理**：本实验额外只保留**面积最大**的检测框（假设主体车辆在画面中占主导地位），然后在原图上绘制绿色矩形显示。

**直观类比**：级联像一场漏斗面试——HR 用一个问题刷掉 90% 的简历，之后每轮面试越来越严格，全部通过才算录用。

**创建一个在图像中检测车辆的函数**

该函数使用预训练的 Haar 级联分类器在输入图像中检测车辆。它首先将图像转换为灰度图，因为 Haar 分类器设计用于单通道图像。

然后它应用 detectMultiScale() 查找车辆区域。为了提高可靠性，它通过只选择最大的边界框来过滤检测结果，假设主体（车辆）在图像中占主导地位。

检测到后，它会在车辆周围绘制绿色矩形，并使用辅助函数（plt_show）显示结果。你可以根据输入图像分辨率调整 scaleFactor、minNeighbors 和 size 参数以提高检测精度。


In [ ]:
def detect_obj(image):
    # 复制原始图像以避免在原图上绘制
    img_copy = image.copy()
    
    # 将图像转换为灰度图，因为 Haar 级联在单通道图像上工作
    gray = cv2.cvtColor(img_copy, cv2.COLOR_BGR2GRAY)

    # 使用放宽参数的 Haar 级联检测对象
    cars = detector.detectMultiScale(
        gray,
        scaleFactor=1.05,  # 稍细的尺度步长，以检测不同大小的对象
        minNeighbors=2,    # 较低的值提高灵敏度（可能会产生更多误检）
        minSize=(30, 30),  # 忽略非常小的检测
        maxSize=(700, 700) # 忽略非常大的检测
    )

    # 如果检测到对象，只保留最大的一个（假设它是主要车辆）
    if len(cars) > 0:
        # 按面积（宽×高）降序排序检测结果
        cars = sorted(cars, key=lambda box: box[2] * box[3], reverse=True)
        # 只保留最大的边界框以避免多个重叠检测
        cars = [cars[0]]

    # 输出检测到的车辆数量（过滤后）
    print(f"检测到：{len(cars)} 辆车")

    # 在检测到的车辆周围绘制绿色矩形
    for (x, y, w, h) in cars:
        cv2.rectangle(img_copy, (x, y), (x + w, y + h), (0, 255, 0), 2)

    # 显示带有检测车辆的图像
    plt_show(img_copy, title="检测到的车辆")


## 图像处理

你正在为车辆检测准备环境，具体包括：

    下载用于车辆检测的预训练 Haar 级联 XML。
    
    使用 OpenCV 的 CascadeClassifier 创建检测器。
    
    从远程 URL 加载测试图像。
    
    使用自定义 plt_show() 函数显示图像。
    
    使用 detect_obj() 函数检测对象。


**从远程 URL 下载 Haar 级联 XML 文件**


从 [andrewssobral](https://raw.githubusercontent.com/andrewssobral/vehicle_detection_haarcascades/master/cars.xml) git 仓库加载预训练分类器，训练需要很长时间但预测很快。


## 🔍 技术原理：`cars.xml` 里到底存了什么

`cars.xml` 并不是神经网络权重，而是**离线训练好的决策参数**，来自 2001 年 Viola & Jones 的经典目标检测方法（OpenCV 将其封装为 `CascadeClassifier`）：

- **Haar 特征**：矩形模板，计算「亮区像素和 − 暗区像素和」。例如「车身与上方背景之间的亮度对比」就是一种可学习的模式。
- **AdaBoost 训练**：用大量「汽车正样本 + 非车负样本」，自动从数以万计的候选特征中筛出最有区分度的几百上千个，并学习每个特征的阈值和权重。
- **级联结构（Cascade）**：特征被组织成多级——靠前的级只用少量简单特征快速排除「明显不是车」的窗口，靠后的级才用更多特征精细判断。

> 训练（选特征、定阈值、排级联顺序）非常耗时，可能要几小时到几天；但训练结果只是一组数字参数，压缩成一个几百 KB 的 XML 文件。这就是「下载即用」的原因：**预测阶段无需训练任何东西**，只是加载别人训练好的「经验」。

In [ ]:
# 定义用于车辆检测的 Haar 级联文件 URL。
# 该文件包含一个用于检测车辆的预训练分类器（特征模式）。
haarcascade_url = 'https://raw.githubusercontent.com/andrewssobral/vehicle_detection_haarcascades/master/cars.xml'

# 定义保存下载 XML 文件的本地文件名。
haar_name = "cars.xml"

# 从 URL 下载 XML 文件并本地保存为 'cars.xml'。
# urllib.request.urlretrieve() 从互联网获取文件。
urllib.request.urlretrieve(haarcascade_url, haar_name)


**使用 `cv2.CascadeClassifier()` 模块在预训练数据集上获取检测器**


## 🔍 技术原理：`CascadeClassifier()` 做了什么

```python
detector = cv2.CascadeClassifier(haar_name)
```

这一行只是把 XML 里的参数（特征模板、阈值、权重、级联结构）**读入内存**，构建出检测器对象。此时还没有对任何图像做计算——真正的检测发生在后面的 `detectMultiScale()`。

In [ ]:
detector = cv2.CascadeClassifier(haar_name)


**读取示例图像**


In [ ]:
# 定义示例图像的 URL（公路上的一辆车）。
image_url = "https://s3.us.cloud-object-storage.appdomain.cloud/cf-courses-data/CognitiveClass/CV0101/Dataset/car-road-behind.jpg"

# 指定下载图像的本地保存名称。
image_name = "car-road-behind.jpg"

# 从给定 URL 下载图像并按指定名称保存。
urllib.request.urlretrieve(image_url, image_name)

# 使用 OpenCV 加载下载的图像（默认以 BGR 格式读取）。
image = cv2.imread(image_name)


**绘制图像**


In [ ]:
plt_show(image)


**在加载的图像上检测对象**


In [ ]:
detect_obj(image)


## 🔍 技术原理：Haar 级联的局限

- 对**视角、光照、遮挡、车型变化**很敏感，容易产生误检和漏检；
- 依赖人工设定的「明暗对比」特征，表达能力有限。

因此实际工程中越来越多地使用深度学习检测器（如本模块的 Faster R-CNN），由神经网络自动学习特征，鲁棒性更好。可以对比下一个 notebook 的检测效果差异。

# 练习 - 上传你的图像


上传你的图像，看看你的车辆是否能被正确检测。
<p><b>如何上传图像：</b></p>
使用上传按钮从本地机器上传图像
<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-CV0101EN-SkillsNetwork/images/instruction.png" width="300"  />
</center>


图像现在会出现在你当前工作的目录中。要在新单元格中读取图像，请使用 <code>cv2.imread</code> 函数。例如，我将 <code>anothercar.jpg</code> 上传到当前工作目录——<code>cv2.imread("anothercar.jpg")</code>。


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-CV0101EN-SkillsNetwork/images/instruction2.png" width="300"  />
</center>


或者使用下面的图像进行测试。


In [ ]:
!wget "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/h3EzuZiidvgdOxPA_yVVWg/car1.jpg"
!wget "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/eKnqJ2xWDdanVdLH6WgOmQ/car2.jpg"
!wget "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/JDjRn_0f5kx9DRT2xv_mew/nocar.jpg"


将下面的 your_uploaded_file 替换为你目录中显示的图像名称。如果你使用的是笔记本中提供的下载图像，则使用 `car1.jpg`、`car2.jpg` 或 `nocar.jpg`


In [ ]:
## 将 "your_uploaded_file" 替换为你的文件名
my_image = cv2.imread("images3.jpg")


In [ ]:
if my_image is None:
    print("Error: image is empty or not loaded properly.")  


在你的图像上运行级联分类器以检测对象。


In [ ]:
detect_obj(my_image)


### 恭喜！你已完成使用 Haar 级联分类器进行车辆检测的实验。
你成功地使用 OpenCV 应用预训练模型检测图像中的车辆。


<h2>作者</h2>


<a href="https://www.linkedin.com/in/aije-egwaikhide/">Aije Egwaikhide</a> 

<a href="https://www.linkedin.com/in/nayefaboutayoun/" target="_blank">Nayef Abou Tayoun</a>

[Sathya Priya](https://www.linkedin.com/in/sathya-priya-06120a17a/) 


<!--<h2>变更日志</h2>-->


<!--<table>
    <tr>
        <th>日期（YYYY-MM-DD）</th>
        <th>版本</th>
        <th>修改人</th>
        <th>变更描述</th>
    </tr>
    <tr>
        <td>2025-06-28</td>
        <td>1.1</td>
        <td>Sathya Priya</td>
        <td>创建并将实验转换为 JupyterCurrent 笔记本 </td>
    </tr>
    <tr>
        <td>2021-04-09</td>
        <td>1.0</td>
        <td>Aije</td>
        <td>更新为新模板</td>
    </tr>
    <tr>
        <td>2021-03-02</td>
        <td>0.2</td>
        <td>Aije</td>
        <td>更新代码和说明并添加练习</td>
    </tr>
    <tr>
        <td>2020-08-18</td>
        <td>0.1</td>
        <td>Nayef</td>
        <td>创建实验原始版本</td>
    </tr>
</table>
-->


<h3 align="center"> &#169; IBM Corporation。保留所有权利。 <h3/>
